# 05 — Đánh giá & so sánh 3 model

**Chủ: Phong Nguyễn** · lý thuyết: `docs/LY_THUYET.md` phần 5 · code: `src/gtsrb/eval/`

Notebook này là nơi **kết luận** của cả dự án được rút ra. Bốn việc:

1. Bảng so sánh chính (top-1, top-5, macro-F1, ECE)
2. ★ **McNemar** — chênh lệch có thật không?
3. Per-class + confusion — lớp nào khó và **vì sao**
4. Calibration — model tự tin đúng mức không?

Chạy `make eval` trước để sinh bảng.

In [ ]:
# Cho notebook thấy src/.  Chạy ô này đầu tiên.
import sys, pathlib
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import os; os.chdir(ROOT)          # mọi đường dẫn trong repo là tương đối từ gốc

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from gtsrb import CLASS_NAMES, NUM_CLASSES
print("torch", torch.__version__, "| MPS", torch.backends.mps.is_available(),
      "| CUDA", torch.cuda.is_available())

In [ ]:
# Tiện ích dùng chung: tìm và nạp lại một run đã train
import glob, json
sys.path.insert(0, str(ROOT / "scripts"))
from evaluate import load_run, test_loader_for
from gtsrb.utils.seed import pick_device

device = pick_device("auto")

def find_runs(prefix=""):
    """Danh sách thư mục run có checkpoint, lọc theo tiền tố tên model."""
    dirs = [pathlib.Path(p) for p in sorted(glob.glob("artifacts/runs/*"))]
    return [d for d in dirs if (d / "best.pt").exists() and d.name.startswith(prefix)]

def latest(prefix):
    runs = find_runs(prefix)
    if not runs:
        raise FileNotFoundError(
            f"Chưa có run nào tên bắt đầu bằng {prefix!r}. Chạy: make train-{prefix[:2]}")
    return runs[-1]

print("Các run đã có:")
for d in find_runs(): print("  ", d.name)

## 1. Bảng so sánh chính

In [ ]:
path = pathlib.Path("reports/tables/main_comparison.csv")
if not path.exists():
    raise SystemExit("Chưa có bảng. Chạy: make eval")
main = pd.read_csv(path)
print(main.to_string(index=False))

**Cách đọc bảng — ba điều phải nói khi trình bày:**

1. **`test_top5` gần như vô nghĩa ở đây.** Top-5 ra đời cho ImageNet 1000 lớp.
   Trên 43 lớp, top-5 nghĩa là "đúng trong 11,6% số lớp" — mọi model tử tế đều ~99,9%.
   Chỉ số bão hoà thì không phân biệt được gì. Nhóm báo cáo vì đề bài yêu cầu,
   nhưng **kết luận dựa trên top-1 và macro-F1**.

2. **`test_macro_f1` là chỉ số chính**, vì nó cho mỗi lớp trọng số bằng nhau
   (dữ liệu mất cân bằng 10,7:1).

3. **Chênh lệch trong bảng chưa chắc có ý nghĩa.** Xem mục 2.

In [ ]:
# Trực quan: macro-F1 vs số tham số
if len(main) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].barh(main["model"], main["test_macro_f1"], color="steelblue")
    axes[0].set_xlabel("test macro-F1"); axes[0].set_xlim(0.9, 1.0)
    axes[0].set_title("Chỉ số chính")
    axes[1].scatter(main["params_m"], main["test_macro_f1"], s=90)
    for r in main.itertuples(index=False):
        axes[1].annotate(r.model, (r.params_m, r.test_macro_f1),
                         textcoords="offset points", xytext=(7, 4), fontsize=8)
    axes[1].set_xlabel("triệu tham số"); axes[1].set_ylabel("test macro-F1")
    axes[1].set_title("Nhiều tham số hơn có tốt hơn?"); axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("Cần >= 2 model để so sánh. Chạy thêm: make train-m2")

## 2. ★ McNemar — "chênh lệch này có THẬT không?"

Hai model chạy trên **cùng** tập test → quan sát **bắt cặp** → dùng t-test hai mẫu
độc lập là **SAI về mặt thống kê**. McNemar là test đúng.

|  | B đúng | B sai |
|---|---|---|
| **A đúng** | n₀₀ | n₀₁ |
| **A sai** | n₁₀ | n₁₁ |

`n₀₀` và `n₁₁` là những ảnh hai model **đồng ý** — chúng **không mang thông tin** về
sự khác biệt. Chỉ hai ô **bất đồng** mới nói được gì.

Trên 12.630 ảnh test, chênh **0,2 điểm top-1 chỉ là ~25 ảnh**.

In [ ]:
mc_path = pathlib.Path("reports/tables/mcnemar.csv")
if mc_path.exists():
    mc = pd.read_csv(mc_path)
    for r in mc.itertuples(index=False):
        print(f"{r.model_a}  vs  {r.model_b}")
        print(f"  n01={r.n01} (chỉ A đúng)   n10={r.n10} (chỉ B đúng)   method={r.method}")
        print(f"  p-value = {r.p_value:.6f}")
        print(f"  -> {r.verdict}\n")
else:
    print("Chưa có reports/tables/mcnemar.csv (cần >= 2 model). Chạy: make eval")

**Cách phát biểu kết luận cho đúng:**

- `p < 0,05` → *"Model X tốt hơn Y **có ý nghĩa thống kê** (p = ...)."*
- `p ≥ 0,05` → *"Chênh lệch **không có ý nghĩa thống kê** (p = ...). Hai model coi như
  tương đương về accuracy; lựa chọn triển khai chuyển sang dựa vào **tốc độ** và
  **robustness**."*

Vế thứ hai là một kết luận **mạnh hơn**, không phải yếu hơn — nó nói được điều mà
bảng accuracy không nói được.

## 3. Per-class — lớp nào khó, và vì sao khó

Hai nguyên nhân phải **tách ra**, không gộp:

- F1 thấp + support **cao** → **hình giống nhau** (vấn đề độ phân giải)
- F1 thấp + support **thấp** → **thiếu dữ liệu**

In [ ]:
for f in sorted(pathlib.Path("reports/tables").glob("per_class_*.csv")):
    name = f.stem.replace("per_class_", "")
    pc = pd.read_csv(f)
    print(f"===== {name} =====")
    print(pc.nsmallest(8, "f1")[["class_id","name","f1","support"]].to_string(index=False))
    print()

In [ ]:
# Tách hai nguyên nhân bằng hình: F1 theo support
files = sorted(pathlib.Path("reports/tables").glob("per_class_*.csv"))
if files:
    fig, axes = plt.subplots(1, len(files), figsize=(5.5 * len(files), 4.2), squeeze=False)
    for ax, f in zip(axes[0], files):
        pc = pd.read_csv(f)
        ax.scatter(pc["support"], pc["f1"], s=40, alpha=0.75)
        for r in pc.nsmallest(6, "f1").itertuples(index=False):
            ax.annotate(str(r.class_id), (r.support, r.f1),
                        textcoords="offset points", xytext=(5, 3), fontsize=8, color="crimson")
        ax.set_xlabel("support (số ảnh test)"); ax.set_ylabel("F1")
        ax.set_title(f.stem.replace("per_class_", "")); ax.grid(alpha=.3)
        ax.axhline(pc["f1"].mean(), ls="--", c="gray", alpha=.6)
    plt.tight_layout(); plt.show()
    print("Nếu các điểm F1 thấp nằm RẢI ĐỀU theo trục support -> không phải do thiếu dữ liệu.")
    print("Khi đó nguyên nhân là HÌNH GIỐNG NHAU -> cách sửa là TĂNG ĐỘ PHÂN GIẢI,")
    print("không phải làm model to hơn.")

In [ ]:
# Các cặp bị nhầm nhiều nhất — bảng này dễ đọc hơn hình confusion 43x43
for f in sorted(pathlib.Path("reports/tables").glob("top_confusions_*.csv")):
    print(f"===== {f.stem.replace('top_confusions_','')} =====")
    print(pd.read_csv(f).to_string(index=False)); print()

**Kết quả đo thật của M1** — ba nguyên nhân, không phải một:

1. **Nhóm biển "HẾT hiệu lực" (lớp 6, 41, 42)** chiếm 3 trong 5 lớp yếu nhất.
   Đây là biển **xám có gạch chéo**, khác nhau rất ít về hình.
   Nhóm **không dự đoán trước** điều này — nó chỉ lộ ra khi đo.
2. **Biển giới hạn tốc độ nhầm lẫn nhau** (3→5, 8→5): cùng hình tròn viền đỏ,
   chỉ khác chữ số, mà ở 48×48 chữ số chỉ còn vài pixel.
3. **Thiếu dữ liệu là nguyên nhân yếu hơn tưởng**: lớp 5 có 630 ảnh test mà vẫn
   nằm trong top 10 yếu nhất; nhiều lớp chỉ 60 ảnh lại F1 = 1,0.

**Dự đoán SAI của nhóm:** nhóm 19/20/21 (đường cong nguy hiểm) **không** xuất hiện
trong top 10 yếu nhất. Nói ra chỗ mình dự đoán sai là một điểm cộng khi bảo vệ.

## 4. Calibration — model tự tin đúng mức không?

`ECE = Σ (|Bₘ|/n) · |acc(Bₘ) − conf(Bₘ)|`

Nó trả lời: *"khi model nói 90% chắc thì nó đúng 90% số lần không?"*

In [ ]:
from gtsrb.eval.calibration import expected_calibration_error
from gtsrb.eval.metrics import evaluate

for run in find_runs():
    model, cfg, _ = load_run(run, device)
    stats = evaluate(model, test_loader_for(cfg), device)
    cal = expected_calibration_error(stats["y_true"], stats["y_prob"])
    conf = stats["y_prob"].max(axis=1)
    chieu = "THIẾU tự tin (underconfident)" if conf.mean() < stats["top1"] \
            else "TỰ TIN THÁI QUÁ (overconfident)"
    print(f"{model.model_name:16s} acc={stats['top1']:.4f}  conf={conf.mean():.4f}  "
          f"ECE={cal['ece']:.4f}  -> {chieu}")
    del model

**★ Kết quả NGƯỢC với lý thuyết thông thường — và đây là một phát hiện của nhóm.**

Lý thuyết (Guo et al. 2017): mạng sâu **tự tin thái quá**, label smoothing sửa được.

Nhóm đo được trên M1: `conf = 0,854 < acc = 0,985` → model **THIẾU tự tin**, ECE = 0,131 (cao).

**Vì sao.** Kết luận của Guo đúng trên ImageNet — bài toán **khó**, accuracy ~76%.
GTSRB **quá dễ**: accuracy 98,5%. Mức tự tin mà label smoothing kéo model về
(≈ `1−ε+ε/K` = 0,9023) **thấp hơn** accuracy thật → nó **sửa quá tay**.

**Hệ quả thực tế:** lập luận "đặt ngưỡng 0,8 thì chuyển cho người lái" **không dùng
được** với model này — ngưỡng đó sẽ loại bỏ rất nhiều dự đoán **đúng**.

**Giả thuyết cho trục ablation 5:** `ε = 0,0` sẽ cho ECE **tốt hơn** trên GTSRB.
Chạy `python scripts/run_ablation.py --axes label_smoothing` để kiểm chứng.

**Bài học:** phải **đo** trên bài toán của mình, không áp kết luận của bài báo vào.